## VALUE INVESTING

In [ ]:
import pandas as pd 
import numpy as np 
import yfinance as yf 
import math
from scipy import stats

In [2]:
tickers= pd.read_csv('top_50_indian_stocks.csv')
tickers.head()

,Ticker,Company Name
0,RELIANCE.NS,Reliance Industries
1,TCS.NS,Tata Consultancy Services
2,HDFCBANK.NS,HDFC Bank
3,INFY.NS,Infosys
4,ICICIBANK.NS,ICICI Bank


In [16]:
import yfinance as yf
import pandas as pd
import numpy as np

def fetch_values_of_stocks(tickers):
    value_cols = [
        "Ticker",
        "Price",
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "EV/EBITDA-Ratio",
        "EV/GP"
    ]

    value_df = pd.DataFrame(columns=value_cols)

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)

            history = stock.history(period="1d")

            if history.empty:
                print(f"{ticker}: no price data")
                continue

            price = history["Close"].iloc[-1]

            financials = stock.financials
            info = stock.info

            pe_ratio = info.get("forwardPE", np.nan)
            pb_ratio = info.get("priceToBook", np.nan)
            ps_ratio = info.get(
                "priceToSalesTrailing12Months",
                np.nan
            )

            ev = info.get("enterpriseValue", np.nan)
            ebitda = info.get("ebitda", np.nan)

            ev_ebitda = (
                ev / ebitda
                if pd.notna(ev)
                and pd.notna(ebitda)
                and ebitda != 0
                else np.nan
            )

            gross_profit = (
                financials.loc["Gross Profit"].iloc[0]
                if "Gross Profit" in financials.index
                else np.nan
            )

            ev_gp = (
                ev / gross_profit
                if pd.notna(ev)
                and pd.notna(gross_profit)
                and gross_profit != 0
                else np.nan
            )

            value_df.loc[len(value_df)] = [
                ticker,
                price,
                pe_ratio,
                pb_ratio,
                ps_ratio,
                ev_ebitda,
                ev_gp
            ]

        except Exception as e:
            print(f"{ticker}: {e}")

    return value_df


tickers_list = tickers["Ticker"].tolist()

df = fetch_values_of_stocks(tickers_list)
df

$TATAMOTORS.NS: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


TATAMOTORS.NS: no price data


,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA-Ratio,EV/GP
0,RELIANCE.NS,1320.000000,18.365366,1.976063,1.689609,12.303381,7.801917
1,TCS.NS,2297.399902,13.887369,7.322927,3.112936,11.163489,6.550987
2,HDFCBANK.NS,742.700012,11.819427,1.950839,4.036405,NaN,NaN
3,INFY.NS,1202.500000,14.670311,5.171048,241.460980,1060.390298,772.398176
4,ICICIBANK.NS,1239.699951,13.629969,2.445235,4.087273,NaN,NaN
5,HINDUNILVR.NS,2084.300049,38.900864,10.051165,7.596409,33.014487,15.589775
6,SBIN.NS,954.099976,9.346737,1.487618,2.338051,NaN,NaN
7,BAJFINANCE.NS,889.049988,17.980310,4.848341,12.611055,NaN,NaN
8,BHARTIARTL.NS,1810.599976,21.700672,7.101227,5.227792,10.591091,8.432559
9,ITC.NS,279.649994,15.737844,4.832717,4.442674,12.390982,7.412521


In [20]:
value_cols = [
        "PE-Ratio",
        "PB-Ratio",
        "PS-Ratio",
        "EV/EBITDA-Ratio",
        "EV/GP"
    ]
for col in value_cols:
    df[col] = df[col].fillna(df[col].mean())

df.info()

<class 'pandas.DataFrame'>
Index: 49 entries, 0 to 48
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Ticker           49 non-null     str    
 1   Price            49 non-null     float64
 2   PE-Ratio         49 non-null     float64
 3   PB-Ratio         49 non-null     float64
 4   PS-Ratio         49 non-null     float64
 5   EV/EBITDA-Ratio  49 non-null     float64
 6   EV/GP            49 non-null     float64
dtypes: float64(6), str(1)
memory usage: 3.1 KB


In [22]:
percentile_metrics={
    "PE-Ratio": "PE-Ratio_Percentile",
    "PB-Ratio": "PB-Ratio_Percentile",
    "PS-Ratio": "PS-Ratio_Percentile",
    "EV/EBITDA-Ratio": "EV/EBITDA-Ratio_Percentile",
    "EV/GP": "EV/GP_Percentile"
}

for metric,percentile in percentile_metrics.items():
    df[percentile] = df[metric].apply(lambda x: stats.percentileofscore(df[metric], x)/100)
df.head()


,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA-Ratio,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA-Ratio_Percentile,EV/GP_Percentile
0,RELIANCE.NS,1320.000000,18.365366,1.976063,1.689609,12.303381,7.801917,0.489796,0.183673,0.142857,0.285714,0.408163
1,TCS.NS,2297.399902,13.887369,7.322927,3.112936,11.163489,6.550987,0.265306,0.755102,0.387755,0.224490,0.326531
2,HDFCBANK.NS,742.700012,11.819427,1.950839,4.036405,75.373246,32.608805,0.163265,0.163265,0.510204,0.877551,0.846939
3,INFY.NS,1202.500000,14.670311,5.171048,241.460980,1060.390298,772.398176,0.326531,0.673469,1.000000,0.979592,1.000000
4,ICICIBANK.NS,1239.699951,13.629969,2.445235,4.087273,75.373246,32.608805,0.244898,0.285714,0.591837,0.877551,0.846939


In [23]:
from statistics import mean

df["Value_Score"] = df[[value for value in percentile_metrics.values()]].mean(axis=1)

df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA-Ratio,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA-Ratio_Percentile,EV/GP_Percentile,Value_Score
0,RELIANCE.NS,1320.000000,18.365366,1.976063,1.689609,12.303381,7.801917,0.489796,0.183673,0.142857,0.285714,0.408163,0.302041
1,TCS.NS,2297.399902,13.887369,7.322927,3.112936,11.163489,6.550987,0.265306,0.755102,0.387755,0.224490,0.326531,0.391837
2,HDFCBANK.NS,742.700012,11.819427,1.950839,4.036405,75.373246,32.608805,0.163265,0.163265,0.510204,0.877551,0.846939,0.512245
3,INFY.NS,1202.500000,14.670311,5.171048,241.460980,1060.390298,772.398176,0.326531,0.673469,1.000000,0.979592,1.000000,0.795918
4,ICICIBANK.NS,1239.699951,13.629969,2.445235,4.087273,75.373246,32.608805,0.244898,0.285714,0.591837,0.877551,0.846939,0.569388
5,HINDUNILVR.NS,2084.300049,38.900864,10.051165,7.596409,33.014487,15.589775,0.816327,0.836735,0.795918,0.571429,0.530612,0.710204
6,SBIN.NS,954.099976,9.346737,1.487618,2.338051,75.373246,32.608805,0.102041,0.102041,0.346939,0.877551,0.846939,0.455102
7,BAJFINANCE.NS,889.049988,17.980310,4.848341,12.611055,75.373246,32.608805,0.448980,0.612245,0.897959,0.877551,0.846939,0.736735
8,BHARTIARTL.NS,1810.599976,21.700672,7.101227,5.227792,10.591091,8.432559,0.612245,0.714286,0.673469,0.163265,0.448980,0.522449
9,ITC.NS,279.649994,15.737844,4.832717,4.442674,12.390982,7.412521,0.387755,0.591837,0.612245,0.306122,0.387755,0.457143


In [24]:
df=df.sort_values(by="Value_Score", ascending=False).reset_index(drop=True)
df

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA-Ratio,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA-Ratio_Percentile,EV/GP_Percentile,Value_Score
0,TITAN.NS,4024.600098,46.883053,22.753923,4.076089,46.466200,59.818971,0.897959,0.979592,0.571429,0.714286,0.979592,0.828571
1,DIVISLAB.NS,6553.500000,46.350800,11.280622,16.474890,50.407838,26.821304,0.877551,0.877551,0.918367,0.734694,0.673469,0.816327
2,PIDILITIND.NS,1457.199951,48.055595,15.527235,10.157658,41.878742,18.167566,0.959184,0.959184,0.836735,0.693878,0.591837,0.808163
3,ADANIGREEN.NS,1446.900024,46.921745,12.780334,18.342947,32.633612,28.838820,0.918367,0.918367,0.938776,0.551020,0.693878,0.804082
4,INFY.NS,1202.500000,14.670311,5.171048,241.460980,1060.390298,772.398176,0.326531,0.673469,1.000000,0.979592,1.000000,0.795918
5,HDFCAMC.NS,2604.699951,29.679087,12.086551,24.155447,30.160250,31.518008,0.755102,0.897959,0.959184,0.510204,0.714286,0.767347
6,ASIANPAINT.NS,2632.399902,45.910637,12.882450,7.091861,37.822773,16.323877,0.857143,0.938776,0.755102,0.673469,0.571429,0.759184
7,BRITANNIA.NS,5157.500000,39.683647,24.329092,6.486554,35.206980,16.153185,0.836735,1.000000,0.734694,0.632653,0.551020,0.751020
8,DMART.NS,4071.800049,57.423670,10.830753,3.858547,51.277616,25.668923,1.000000,0.857143,0.469388,0.755102,0.653061,0.746939
9,BAJFINANCE.NS,889.049988,17.980310,4.848341,12.611055,75.373246,32.608805,0.448980,0.612245,0.897959,0.877551,0.846939,0.736735


In [25]:
df.head(10)

,Ticker,Price,PE-Ratio,PB-Ratio,PS-Ratio,EV/EBITDA-Ratio,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA-Ratio_Percentile,EV/GP_Percentile,Value_Score
0,TITAN.NS,4024.600098,46.883053,22.753923,4.076089,46.466200,59.818971,0.897959,0.979592,0.571429,0.714286,0.979592,0.828571
1,DIVISLAB.NS,6553.500000,46.350800,11.280622,16.474890,50.407838,26.821304,0.877551,0.877551,0.918367,0.734694,0.673469,0.816327
2,PIDILITIND.NS,1457.199951,48.055595,15.527235,10.157658,41.878742,18.167566,0.959184,0.959184,0.836735,0.693878,0.591837,0.808163
3,ADANIGREEN.NS,1446.900024,46.921745,12.780334,18.342947,32.633612,28.838820,0.918367,0.918367,0.938776,0.551020,0.693878,0.804082
4,INFY.NS,1202.500000,14.670311,5.171048,241.460980,1060.390298,772.398176,0.326531,0.673469,1.000000,0.979592,1.000000,0.795918
5,HDFCAMC.NS,2604.699951,29.679087,12.086551,24.155447,30.160250,31.518008,0.755102,0.897959,0.959184,0.510204,0.714286,0.767347
6,ASIANPAINT.NS,2632.399902,45.910637,12.882450,7.091861,37.822773,16.323877,0.857143,0.938776,0.755102,0.673469,0.571429,0.759184
7,BRITANNIA.NS,5157.500000,39.683647,24.329092,6.486554,35.206980,16.153185,0.836735,1.000000,0.734694,0.632653,0.551020,0.751020
8,DMART.NS,4071.800049,57.423670,10.830753,3.858547,51.277616,25.668923,1.000000,0.857143,0.469388,0.755102,0.653061,0.746939
9,BAJFINANCE.NS,889.049988,17.980310,4.848341,12.611055,75.373246,32.608805,0.448980,0.612245,0.897959,0.877551,0.846939,0.736735
